In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer, StandardScaler
import joblib
import os
from datetime import datetime

print("🚀 CIC-IDS-2018 CHUNK-BASED SCALING SCRIPT")
print("=" * 80)

# ============================================================================
# CONFIGURATION
# ============================================================================

# File paths
INPUT_FILE = 'archive/full_df_binary_labels.csv'  # Change this to your actual file
OUTPUT_DIR = 'archive'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'cicids2018_scaled.csv')
SCALER_FILE = os.path.join(OUTPUT_DIR, 'fitted_scalers.pkl')

# Chunk size - adjust based on your RAM (smaller = safer but slower)
CHUNK_SIZE = 50000  # Process 50k rows at a time

# Feature groups
TARGET_COL = 'Label_Binary'

# Features for StandardScaler (low skew)
STANDARD_FEATURES = ['ACK Flag Cnt', 'PSH Flag Cnt', 'Fwd Seg Size Min']

# Features for PowerTransformer (all others)
POWER_FEATURES = [
    'Flow IAT Mean', 'Fwd IAT Mean', 'Idle Min', 'Pkt Len Var', 'Fwd IAT Tot',
    'Flow Duration', 'Idle Max', 'Idle Mean', 'Flow IAT Std', 'Fwd IAT Std',
    'Flow IAT Min', 'Fwd IAT Min', 'Fwd IAT Max', 'Flow IAT Max', 'TotLen Fwd Pkts',
    'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Tot Bwd Pkts', 'Bwd Header Len',
    'TotLen Bwd Pkts', 'Subflow Bwd Byts', 'Down/Up Ratio', 'Fwd Act Data Pkts',
    'Subflow Fwd Pkts', 'Tot Fwd Pkts', 'Fwd Header Len', 'Flow Byts/s',
    'Bwd IAT Min', 'Bwd IAT Mean', 'Bwd Pkts/s', 'Fwd Pkts/s', 'Fwd Pkt Len Mean',
    'Fwd Seg Size Avg', 'Flow Pkts/s', 'Bwd IAT Std', 'Bwd IAT Max', 'Fwd Pkt Len Max',
    'URG Flag Cnt', 'Pkt Len Mean', 'Bwd Seg Size Avg', 'Bwd Pkt Len Mean',
    'Pkt Size Avg', 'Bwd IAT Tot', 'Init Fwd Win Byts', 'Fwd Pkt Len Std',
    'Init Bwd Win Byts', 'Pkt Len Max', 'Dst Port', 'Pkt Len Std',
    'Bwd Pkt Len Max', 'RST Flag Cnt', 'ECE Flag Cnt', 'Bwd Pkt Len Std', 'Protocol'
]

# ============================================================================
# CREATE OUTPUT DIRECTORY
# ============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

# ============================================================================
# STEP 1: FIT SCALERS ON SAMPLE DATA
# ============================================================================

print("\n" + "=" * 80)
print("STEP 1: FITTING SCALERS ON SAMPLE DATA")
print("=" * 80)

# Read a sample to fit the scalers
print(f"\n📖 Reading sample data for fitting scalers...")
sample_size = min(100000, CHUNK_SIZE * 5)  # Use 100k rows or 5 chunks

try:
    df_sample = pd.read_csv(INPUT_FILE, nrows=sample_size)
    print(f"✅ Loaded {len(df_sample):,} rows for fitting")
except Exception as e:
    print(f"❌ Error reading file: {e}")
    print("💡 Make sure to set INPUT_FILE to your actual dataset path")
    exit()

# Verify columns exist
missing_cols = []
for col in STANDARD_FEATURES + POWER_FEATURES + [TARGET_COL]:
    if col not in df_sample.columns:
        missing_cols.append(col)

if missing_cols:
    print(f"\n⚠️  WARNING: These columns are missing from your dataset:")
    print(f"   {missing_cols}")
    print(f"\n📋 Available columns in your dataset:")
    print(f"   {df_sample.columns.tolist()}")

    # Auto-detect features
    print(f"\n🔧 Auto-detecting features...")
    TARGET_COL = 'Label_Binary' if 'Label_Binary' in df_sample.columns else df_sample.columns[-1]
    all_numeric = df_sample.select_dtypes(include=[np.number]).columns.tolist()
    all_numeric = [col for col in all_numeric if col != TARGET_COL]

    STANDARD_FEATURES = []
    POWER_FEATURES = all_numeric

    print(f"   ✅ Target: {TARGET_COL}")
    print(f"   ✅ Features to scale: {len(POWER_FEATURES)}")

# Initialize scalers
print(f"\n🔧 Initializing scalers...")
scaler_power = PowerTransformer(method='yeo-johnson', standardize=True)
scaler_standard = StandardScaler()

# Fit scalers
print(f"\n⚙️  Fitting PowerTransformer on {len(POWER_FEATURES)} features...")
if POWER_FEATURES:
    scaler_power.fit(df_sample[POWER_FEATURES])
    print(f"   ✅ PowerTransformer fitted")

if STANDARD_FEATURES:
    print(f"\n⚙️  Fitting StandardScaler on {len(STANDARD_FEATURES)} features...")
    scaler_standard.fit(df_sample[STANDARD_FEATURES])
    print(f"   ✅ StandardScaler fitted")

# Save fitted scalers
scalers_dict = {
    'power_transformer': scaler_power,
    'standard_scaler': scaler_standard if STANDARD_FEATURES else None,
    'power_features': POWER_FEATURES,
    'standard_features': STANDARD_FEATURES,
    'target_col': TARGET_COL,
    'fitted_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

joblib.dump(scalers_dict, SCALER_FILE)
print(f"\n💾 Scalers saved to: {SCALER_FILE}")

# Clean up
del df_sample

# ============================================================================
# STEP 2: PROCESS DATASET IN CHUNKS
# ============================================================================

print("\n" + "=" * 80)
print("STEP 2: PROCESSING FULL DATASET IN CHUNKS")
print("=" * 80)

# Count total rows
print(f"\n📊 Counting total rows...")
total_rows = sum(1 for _ in open(INPUT_FILE)) - 1  # -1 for header
print(f"   Total rows: {total_rows:,}")
total_chunks = (total_rows // CHUNK_SIZE) + 1
print(f"   Total chunks: {total_chunks}")
print(f"   Chunk size: {CHUNK_SIZE:,} rows")

# Process chunks
print(f"\n🔄 Processing chunks...")
first_chunk = True
rows_processed = 0

chunk_iterator = pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)

for chunk_num, chunk in enumerate(chunk_iterator, 1):

    # Progress indicator
    rows_processed += len(chunk)
    progress_pct = (rows_processed / total_rows) * 100
    print(f"\n📦 Chunk {chunk_num}/{total_chunks} | Rows: {rows_processed:,}/{total_rows:,} ({progress_pct:.1f}%)")

    # Separate target from features
    if TARGET_COL in chunk.columns:
        target = chunk[[TARGET_COL]]
        features = chunk.drop(columns=[TARGET_COL])
    else:
        target = None
        features = chunk

    # Transform PowerTransformer features
    if POWER_FEATURES:
        power_cols = [col for col in POWER_FEATURES if col in features.columns]
        features[power_cols] = scaler_power.transform(features[power_cols])

    # Transform StandardScaler features
    if STANDARD_FEATURES:
        standard_cols = [col for col in STANDARD_FEATURES if col in features.columns]
        features[standard_cols] = scaler_standard.transform(features[standard_cols])

    # Recombine with target
    if target is not None:
        chunk_scaled = pd.concat([features, target], axis=1)
    else:
        chunk_scaled = features

    # Write to CSV
    if first_chunk:
        chunk_scaled.to_csv(OUTPUT_FILE, index=False, mode='w')
        first_chunk = False
        print(f"   ✅ Created output file: {OUTPUT_FILE}")
    else:
        chunk_scaled.to_csv(OUTPUT_FILE, index=False, mode='a', header=False)
        print(f"   ✅ Appended to output file")

    # Memory cleanup
    del chunk, features, chunk_scaled
    if target is not None:
        del target

print(f"\n✅ Processing complete!")

# ============================================================================
# STEP 3: VERIFICATION
# ============================================================================

print("\n" + "=" * 80)
print("STEP 3: VERIFICATION")
print("=" * 80)

print(f"\n📊 Verifying output file...")
df_verify = pd.read_csv(OUTPUT_FILE, nrows=5)

print(f"✅ Output file exists: {OUTPUT_FILE}")
print(f"✅ Shape: {len(pd.read_csv(OUTPUT_FILE)):,} rows x {len(df_verify.columns)} columns")

print(f"\n📋 First 5 rows of scaled data:")
print(df_verify)

# Check statistics
print(f"\n📊 Quick statistics (first 10,000 rows):")
df_stats = pd.read_csv(OUTPUT_FILE, nrows=10000)

for feature in POWER_FEATURES[:5]:  # Check first 5 features
    if feature in df_stats.columns:
        print(f"\n• {feature}:")
        print(f"  Mean: {df_stats[feature].mean():.6f} (should be ≈ 0)")
        print(f"  Std:  {df_stats[feature].std():.6f} (should be ≈ 1)")
        print(f"  Skew: {df_stats[feature].skew():.6f}")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("🎉 TRANSFORMATION COMPLETE!")
print("=" * 80)

print(f"\n📁 Output files:")
print(f"   • Scaled data: {OUTPUT_FILE}")
print(f"   • Fitted scalers: {SCALER_FILE}")

print(f"\n📊 Summary:")
print(f"   • Total rows processed: {rows_processed:,}")
print(f"   • Features scaled with PowerTransformer: {len(POWER_FEATURES)}")
if STANDARD_FEATURES:
    print(f"   • Features scaled with StandardScaler: {len(STANDARD_FEATURES)}")
print(f"   • Target column preserved: {TARGET_COL}")

print(f"\n💡 Next steps:")
print(f"   1. Load scaled data: df = pd.read_csv('{OUTPUT_FILE}')")
print(f"   2. For test data, use the saved scalers:")
print(f"      scalers = joblib.load('{SCALER_FILE}')")
print(f"      X_test_scaled = scalers['power_transformer'].transform(X_test)")

print("\n✅ Done!")

🚀 CIC-IDS-2018 CHUNK-BASED SCALING SCRIPT
✅ Output directory: archive

STEP 1: FITTING SCALERS ON SAMPLE DATA

📖 Reading sample data for fitting scalers...
✅ Loaded 100,000 rows for fitting

🔧 Initializing scalers...

⚙️  Fitting PowerTransformer on 54 features...
   ✅ PowerTransformer fitted

⚙️  Fitting StandardScaler on 3 features...
   ✅ StandardScaler fitted

💾 Scalers saved to: archive\fitted_scalers.pkl

STEP 2: PROCESSING FULL DATASET IN CHUNKS

📊 Counting total rows...
   Total rows: 16,232,943
   Total chunks: 325
   Chunk size: 50,000 rows

🔄 Processing chunks...

📦 Chunk 1/325 | Rows: 50,000/16,232,943 (0.3%)
   ✅ Created output file: archive\cicids2018_scaled.csv

📦 Chunk 2/325 | Rows: 100,000/16,232,943 (0.6%)
   ✅ Appended to output file

📦 Chunk 3/325 | Rows: 150,000/16,232,943 (0.9%)
   ✅ Appended to output file

📦 Chunk 4/325 | Rows: 200,000/16,232,943 (1.2%)
   ✅ Appended to output file

📦 Chunk 5/325 | Rows: 250,000/16,232,943 (1.5%)
   ✅ Appended to output file

📦 